# ETL Data Pipeline — Open Food Facts

**Extract** 50,000 food products from the [Open Food Facts API](https://world.openfoodfacts.org/),
**Transform** the raw JSON into a clean, analysis-ready dataset with engineered nutrition features,
and **Load** everything into a normalised SQLite database.

**Pipeline overview:**

```
Open Food Facts API  ──►  JSON  ──►  CSV (filtered & cleaned)  ──►  SQLite database
     (Extract)                          (Transform)                      (Load)
```

### Database schema

| Table | Description |
|-------|-------------|
| **Brand** | Unique brand names (brand_id, brand_name) |
| **Category** | Unique food categories (category_id, category_name) |
| **Country** | Unique countries (country_id, country_name) |
| **Product** | Products with nutrition data and FK to Brand & Category |
| **Product_Country** | Many-to-many junction table (product_id ↔ country_id) |

## 0 — Imports

In [ ]:
import pandas as pd
import json
import csv
import requests
import sqlite3
import time
import matplotlib.pyplot as plt
import seaborn as sns

## 1 — Extract

Paginate through the Open Food Facts search API (v2), requesting 100 products per
page sorted by popularity (`unique_scans_n`).  We fetch up to 500 pages
(≈ 50,000 products) and save the raw response as a local JSON file so the
extraction step only needs to run once.

In [ ]:
all_products = []
page = 1

while page <= 500:
    url = (
        "https://world.openfoodfacts.org/api/v2/search"
        "?fields=product_name,brands_tags,categories_tags,"
        "countries_tags,nutriments"
        "&sort_by=unique_scans_n&page_size=100"
        f"&page={page}"
    )
    print(f"Fetching page {page} …")
    response = requests.get(url)
    data = response.json()
    products = data.get("products", [])

    if not products:
        print("No more products found — stopping.")
        break

    all_products.extend(products)
    page += 1
    time.sleep(0.2)          # respect the API rate limit

# Persist raw data so we don't need to re-fetch
with open("openfoodfacts_data.json", "w", encoding="utf-8") as f:
    json.dump(all_products, f, indent=2, ensure_ascii=False)

print(f"\nFetched {len(all_products):,} products in total.")

## 2 — Transform

### 2.1 JSON → CSV

Not every product record has complete data.  We keep only products that have
all three nutrient values **and** at least one brand, category and country tag.

In [ ]:
with open("openfoodfacts_data.json", "r", encoding="utf-8") as f:
    all_products = json.load(f)

valid_products = []

for product in all_products:
    nutrients = product.get("nutriments", {})
    sugar    = nutrients.get("sugars_100g")
    energy   = nutrients.get("energy_100g")
    protein  = nutrients.get("proteins_100g")
    brands     = product.get("brands_tags", [])
    categories = product.get("categories_tags", [])
    countries  = product.get("countries_tags", [])

    if (sugar is not None
        and energy is not None
        and protein is not None
        and brands and categories and countries):
        valid_products.append({
            "product_name": product.get("product_name"),
            "brand":    brands[0],
            "category": categories[0],
            "country":  countries[0],
            "sugar_100g":   sugar,
            "energy_100g":  energy,
            "protein_100g": protein,
        })

# Write to CSV
with open("valid_products_full.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "product_name", "brand", "category", "country",
        "sugar_100g", "energy_100g", "protein_100g",
    ])
    writer.writeheader()
    writer.writerows(valid_products)

print(f"CSV created with {len(valid_products):,} valid products.")

### 2.2 First look at the data

In [ ]:
df = pd.read_csv("valid_products_full.csv")
print(f"Shape: {df.shape}")
print(f"\nNull values:\n{df.isnull().sum()}")
print(f"\nDuplicates: {df.duplicated().sum()}")
df.head()

### 2.3 Clean missing values & duplicates

Drop rows where `product_name` or `brand` is missing, remove exact
duplicates, and filter out non-food categories (cosmetics, cleaning, baby).

In [ ]:
df = df[df["product_name"].notnull()].copy()
df = df[df["brand"].notnull()]
df.drop_duplicates(inplace=True)

# Remove non-food categories
df = df[~df["category"].str.contains("cosmetic|cleaning|baby", na=False)]

print(f"Rows after cleaning: {len(df):,}")

### 2.4 Text normalisation

Remove the `en:` prefix that the API adds to category and country tags,
convert text columns to lowercase, and replace hyphens with spaces.

In [ ]:
# Remove "en:" prefix
df["category"] = df["category"].str.replace("en:", "", regex=False)
df["country"]  = df["country"].str.replace("en:", "", regex=False)

# Lowercase, strip whitespace, replace hyphens
for col in ["brand", "category", "country"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace("-", " ", regex=False)
    )

# Clean product names
df["product_name"] = (
    df["product_name"]
    .astype(str)
    .str.strip()
    .str.replace(r"[^\w\s\-]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.title()
)

# Remove rows where product_name is just digits or empty
df = df[df["product_name"].str.strip() != ""]
df = df[~df["product_name"].str.match(r"^\d+$", na=False)]

# Round nutrient values
df[["sugar_100g", "energy_100g", "protein_100g"]] = (
    df[["sugar_100g", "energy_100g", "protein_100g"]]
    .map(lambda x: round(x, 1) if pd.notnull(x) else x)
)

print(f"Rows after normalisation: {len(df):,}")
df.head()

### 2.5 Feature engineering

Create ordinal nutrition-level columns and a binary `is_healthy` flag.

In [ ]:
def classify_sugar(sugar):
    if sugar < 10:   return 0   # low
    elif sugar <= 20: return 1   # medium
    else:             return 2   # high

def classify_energy(energy):
    if energy < 100:    return 0
    elif energy <= 400:  return 1
    else:                return 2

def classify_protein(protein):
    if protein < 5:     return 0
    elif protein <= 15:  return 1
    else:                return 2

df["sugar_level"]   = df["sugar_100g"].apply(classify_sugar)
df["energy_level"]  = df["energy_100g"].apply(classify_energy)
df["protein_level"] = df["protein_100g"].apply(classify_protein)

# A product is flagged "healthy" if sugar < 10 g AND protein > 5 g per 100 g
df["is_healthy"] = ((df["sugar_100g"] < 10) & (df["protein_100g"] > 5)).astype(int)

print(f"Healthy products: {df['is_healthy'].sum():,}  "
      f"({df['is_healthy'].mean():.1%} of total)")

### 2.6 Add product ID and save clean CSV

In [ ]:
df.reset_index(drop=True, inplace=True)
df.insert(0, "product_id", range(1, len(df) + 1))

df.to_csv("clean_products.csv", index=False)

print(f"Final dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 3 — Load

### 3.1 Create the SQLite schema

Five tables with foreign-key constraints:

```
Brand  ◄──  Product  ──►  Category
                │
                └──►  Product_Country  ◄──  Country
```

In [ ]:
conn = sqlite3.connect("food_data.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS Brand (
    brand_id    INTEGER PRIMARY KEY,
    brand_name  TEXT UNIQUE
)""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Category (
    category_id   INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE
)""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Country (
    country_id   INTEGER PRIMARY KEY,
    country_name TEXT UNIQUE
)""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Product (
    product_id    INTEGER PRIMARY KEY,
    product_name  TEXT,
    energy_100g   REAL,
    sugar_100g    REAL,
    protein_100g  REAL,
    sugar_level   INTEGER,
    energy_level  INTEGER,
    protein_level INTEGER,
    is_healthy    INTEGER,
    brand_id      INTEGER,
    category_id   INTEGER,
    FOREIGN KEY (brand_id)    REFERENCES Brand(brand_id),
    FOREIGN KEY (category_id) REFERENCES Category(category_id)
)""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Product_Country (
    product_id INTEGER,
    country_id INTEGER,
    PRIMARY KEY (product_id, country_id),
    FOREIGN KEY (product_id) REFERENCES Product(product_id),
    FOREIGN KEY (country_id) REFERENCES Country(country_id)
)""")

conn.commit()
print("Schema created successfully.")

### 3.2 Populate the database

In [ ]:
df = pd.read_csv("clean_products.csv")

# ── Insert dimension tables ──────────────────────────────────────────────────
for brand in df["brand"].dropna().unique():
    cursor.execute("INSERT OR IGNORE INTO Brand (brand_name) VALUES (?)", (brand,))

for category in df["category"].dropna().unique():
    cursor.execute("INSERT OR IGNORE INTO Category (category_name) VALUES (?)", (category,))

for country in df["country"].dropna().unique():
    cursor.execute("INSERT OR IGNORE INTO Country (country_name) VALUES (?)", (country,))

conn.commit()

# ── Insert products ──────────────────────────────────────────────────────────
for _, row in df.iterrows():
    if pd.isnull(row["product_name"]):
        continue

    cursor.execute("SELECT brand_id FROM Brand WHERE brand_name = ?",
                   (row["brand"],))
    brand_id = (cursor.fetchone() or [None])[0]

    cursor.execute("SELECT category_id FROM Category WHERE category_name = ?",
                   (row["category"],))
    category_id = (cursor.fetchone() or [None])[0]

    cursor.execute("""
        INSERT INTO Product (
            product_name, energy_100g, sugar_100g, protein_100g,
            sugar_level, energy_level, protein_level, is_healthy,
            brand_id, category_id
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        row["product_name"],
        float(row["energy_100g"]),
        float(row["sugar_100g"]),
        float(row["protein_100g"]),
        int(row["sugar_level"]),
        int(row["energy_level"]),
        int(row["protein_level"]),
        int(row["is_healthy"]),
        brand_id,
        category_id,
    ))

conn.commit()

# ── Link products to countries ───────────────────────────────────────────────
cursor.execute("SELECT product_id, product_name FROM Product")
product_dict = {name: pid for pid, name in cursor.fetchall()}

for _, row in df.iterrows():
    product_id = product_dict.get(row["product_name"])
    if product_id is None:
        continue

    cursor.execute("SELECT country_id FROM Country WHERE country_name = ?",
                   (row["country"],))
    result = cursor.fetchone()
    if result:
        cursor.execute(
            "INSERT OR IGNORE INTO Product_Country (product_id, country_id) "
            "VALUES (?, ?)",
            (product_id, result[0]),
        )

conn.commit()
print("All data inserted successfully.")

### 3.3 Verify the load

In [ ]:
tables = ["Brand", "Category", "Country", "Product", "Product_Country"]
for t in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"{t:20s} → {cursor.fetchone()[0]:,} rows")

In [ ]:
# Sample query: top 10 brands by product count
cursor.execute("""
    SELECT b.brand_name, COUNT(*) AS n_products
    FROM Product p
    JOIN Brand b ON p.brand_id = b.brand_id
    GROUP BY b.brand_name
    ORDER BY n_products DESC
    LIMIT 10
""")

print("Top 10 brands by product count:\n")
for brand, count in cursor.fetchall():
    print(f"  {brand:30s} {count:,}")

In [ ]:
# Sample query: nutrition summary by category (top 8)
cursor.execute("""
    SELECT c.category_name,
           COUNT(*)              AS n_products,
           ROUND(AVG(p.sugar_100g), 1)  AS avg_sugar,
           ROUND(AVG(p.energy_100g), 0) AS avg_energy,
           ROUND(AVG(p.protein_100g), 1) AS avg_protein
    FROM Product p
    JOIN Category c ON p.category_id = c.category_id
    GROUP BY c.category_name
    ORDER BY n_products DESC
    LIMIT 8
""")

print("Nutrition summary by category:\n")
print(f"{'Category':45s} {'#':>6s}  {'Sugar':>6s}  {'Energy':>7s}  {'Protein':>7s}")
print("-" * 80)
for row in cursor.fetchall():
    print(f"{row[0]:45s} {row[1]:6,}  {row[2]:6.1f}  {row[3]:7.0f}  {row[4]:7.1f}")

In [ ]:
conn.close()
print("Database connection closed.")